# 09 · Chunk Metadata：切片的身份信息

> 没有 metadata 的 chunk 只是“一串字”；有了 metadata，才能过滤、溯源、引用、权限控制。

**本文件覆盖知识点**：metadata 设计 / Metadata Filtering / Metadata Index / Metadata Retrieval / Source Tracking / Citation

典型 metadata 结构：
```json
{"document_id":"123", "page":10, "section":"向量数据库", "title":"缓存穿透", "chunk_id":"123_10_3"}
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. metadata 解决的四件事

| 需求 | 靠什么 | 例子 |
|------|--------|------|
| **Source Tracking** | 知道内容来自哪 | `document_id` / `filepath` |
| **定位** | 快速找到原位置 | `page` / `section` / `chunk_id` |
| **过滤(Filtering)** | 检索前缩小范围 | `date>=2024` 或 `author=A` |
| **引用/权限** | 回答标注出处、按权限隔离 | `source` + ACL 字段 |

在向量库层面，metadata 分两类用途：
- **Metadata Filter**：先按字段过滤，再在剩余子集上做向量检索（预筛）；
- **Metadata Index**：为这些字段建索引，让过滤本身也快。

In [ ]:
# 演示：把 metadata 跟 chunk 绑在一起，形成结构化记录
def make_chunk(text, doc_id, page, section, seq):
    """生成一个带身份证(chunk_id)的 chunk"""
    return {
        'text': text,
        'metadata': {
            'document_id': doc_id,
            'page': page,
            'section': section,
            'chunk_id': f'{doc_id}_{page}_{seq}',   # 全局唯一、可排序
        },
    }

c1 = make_chunk('向量数据库的核心是相似度检索', 'doc-001', 3, '向量数据库', 0)
print(c1['metadata'])
print('\nchunk_id 可解析: 文档', c1['metadata']['chunk_id'].split('_')[0], '| 页', c1['metadata']['chunk_id'].split('_')[1])

## 2. Metadata Filtering 实战思路

```text
用户(有权限 A/B): “请查 A、B 部门 2024 年后的规范”
         │
         ▼
预筛:  dept IN (A,B)  AND  year >= 2024      ← 纯结构过滤
         │
         ▼
向量检索在预筛后的子集上进行                          ← 语义精排
```

好处：既提升相关性（排除噪声域），又实现多租户隔离（第 36 课权限），还能降低检索规模。

In [ ]:
# 知识点·真调说明：Metadata Filtering —— 不做预筛时新旧两版资料“打架”，按版本年份过滤后回答才干净
# metadata 预筛（这里按版本年份）把噪声资料挡在生成之前，正是“先过滤、再语义检索”的价值。
print('① 不做 metadata 过滤 —— 新旧两版都被拼进上下文，模型只能报出矛盾')
_llm_live(
    prompt="""请只依据下面两条资料回答：按公司现行制度，单笔 8 万元的采购需要谁审批？
资料A（2023 版·采购制度）：「单笔采购金额超过 5 万元须总经理审批。」
资料B（2024 版·采购制度）：「自 2024 年 1 月起，单笔采购 20 万元及以下由部门负责人审批，超过 20 万元须总经理审批；5 万元旧标准不再适用。」""",
    system='你是企业内审助手，只依据给定资料作答；资料相互矛盾时如实指出矛盾，不要自行判断哪个版本有效。',
    fallback="""资料存在矛盾：A 说超过 5 万元须总经理审批（8 万元 → 总经理）；B 说 2024 年起 20 万元及以下由部门负责人审批（8 万元 → 部门负责人）。仅凭这两条无法判定现行标准。""",
    temperature=0.1,
)
print()
print('② 先做 metadata 过滤（year>=2024，剔除 2023 旧版）再作答 —— 结论干净、无歧义')
_llm_live(
    prompt="""请只依据下面给出的资料回答：按公司现行制度，单笔 8 万元的采购需要谁审批？
资料（2024 版·采购制度）：「自 2024 年 1 月起，单笔采购 20 万元及以下由部门负责人审批，超过 20 万元须总经理审批；5 万元旧标准不再适用。」""",
    system='你是企业内审助手，只依据给定资料作答。',
    fallback="""8 万元在 20 万元及以下，由部门负责人审批即可，无需总经理审批。""",
    temperature=0.1,
)
print()
print('→ 全库语义检索容易把“看起来相关实则过时”的旧版一起召回，把矛盾带进回答；')
print('  metadata 预筛（dept/year/权限）在向量检索前先剔除噪声，既稳回答、又能实现多租户隔离（第 36 课）。')

## 3. 从 metadata 到 Citation

回答要“可溯源”，关键是把检索命中的 chunk_id / source / page 一路带到生成层：

```text
检索到 chunk(doc=星云产品手册, page=2)
   → 把它编号 [来源1] 注入 prompt
   → 模型说“支持私有化部署[来源1]”
   → 前端把 [来源1] 渲染成可点击卡片: 星云产品手册 p2
```

> 一句话：metadata 是从答案指回原文的桥，没有它 Citation 无从谈起。



In [ ]:
# 知识点·真调说明：Citation —— 给每条资料带上 chunk_id/页码 的编号，模型才能给出“可点击的出处”
# metadata 里存的 document_id/page 正是“从答案指回原文”这座桥。
_FB = """支持私有化部署[1]。私有化部署需客户自备服务器，并提供至少 8 核 16G 资源[2]。"""
out = _llm_live(
    prompt="""请依据下面两条资料回答“星云客服机器人支持私有化部署吗”，并在每句结论后用 [1] 或 [2] 标出依据。
资料[1]（《星云产品手册》第2页）：「星云客服机器人支持私有化部署，也可选公有云 SaaS。」
资料[2]（《星云产品手册》第5页）：「私有化部署需要客户自备服务器，并提供至少 8 核 16G 资源。」""",
    system='你是客服问答助手，给结论标注来源编号 [1]/[2]，没有对应资料依据的内容不要写。',
    fallback=_FB,
    temperature=0.1,
)
import re as _re
if out is None:
    out = _FB
    print('（以上为未配置 Key 时的固定样例；下面照样例演示“出处标记”检查）')
marks = _re.findall(r'\[\d+\]', out)
print('模型回答里出现的来源标记：', marks)
print('→ 模型能写出 [1]/[2]，靠的是每条资料前面带的 document/page 编号——也就是 metadata 存的字段。')
print('  没有 metadata 一路带到生成层，模型根本不知道某句话出自哪一页，Citation（可点击引用）也就无从谈起。')

## 小结

- chunk = 正文 + metadata（文档/页/节/唯一 id）；
- metadata 支撑过滤、索引、溯源、引用、权限；
- 设计原则：把“检索后还要用”的稳定属性都放进 metadata，别等上线再补。